In [85]:
# legacy_linking_v7 — CELL L1: Setup and load raw Excel inputs

import pandas as pd
from pathlib import Path

BASE    = Path("/Users/davekokel/Projects/carp_v2")
RAW     = BASE / "seed_kits" / "legacy_wrangling" / "raw"
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

roi_paths_path     = RAW / "2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx"
imaging_sheet_path = RAW / "2025-11-13-124226-imaging_sheet.xlsx"

# Raw ROI paths
df_roi_paths = pd.read_excel(roi_paths_path)

# Imaging sheet is a single worksheet with ROI + mount/metadata
df_imaging = pd.read_excel(imaging_sheet_path)



In [86]:
# CELL L2 — standardize ROI paths

df_roi_paths["roi_dir"]  = df_roi_paths["roi_dir"].astype(str).str.strip()
df_roi_paths["roi_name"] = df_roi_paths["roi_name"].astype(str).str.strip()
ROI_KEY = "roi_dir"

In [87]:
# CELL L3 — mount_dir_norm in both tables + merge on mount_dir_norm

import re

def norm_path(p: str) -> str:
    if p is None or (isinstance(p, float) and pd.isna(p)):
        return None
    s = str(p).replace("\\", "/")
    s = re.sub(r"^[A-Za-z]:", "", s)          # strip X: drive
    s = s.replace("/clusterfs/vast", "")
    s = s.replace("/clusterfs/nvme", "")
    s = s.replace("/clusterfs/nvme2", "")
    s = re.sub(r"/+", "/", s)
    return s.rstrip("/")

# imaging: mount-level path
df_imaging["mount_dir_norm"] = df_imaging["Data location"].apply(norm_path)

# ROI: mount-dir = roi_dir without the last segment
def roi_mount_dir(path: str) -> str:
    s = norm_path(path)
    if s is None:
        return None
    parts = s.split("/")
    if len(parts) <= 1:
        return s
    # drop last segment (roiX_...)
    return "/".join(parts[:-1])

df_roi_paths["mount_dir_norm"] = df_roi_paths["roi_dir"].apply(roi_mount_dir)

df_roi_paths[["roi_dir","mount_dir_norm"]].head()
df_imaging[["Data location","mount_dir_norm","ZF female genotype","ZF male genotype"]].head()

,Data location,mount_dir_norm,ZF female genotype,ZF male genotype
0,X:\abcabc\Korra_Foundation\20251112_mem-mito,/abcabc/Korra_Foundation/20251112_mem-mito,casper,memrane tdmScarlet3S2 (F1 of allele 325)
1,X:\abcabc\Exploratory_fish\20251112_mem-moesin,/abcabc/Exploratory_fish/20251112_mem-moesin,casper,memrane tdmScarlet3S2 (F1 of allele 325)
2,X:\abcabc\Aang_Foundation\20251107_mem-mito,/abcabc/Aang_Foundation/20251107_mem-mito,casper,memrane tdmScarlet3S2 (F1 of allele 325)
3,X:\abcabc\Korra_Foundation\20251110_mem-histone,/abcabc/Korra_Foundation/20251110_mem-histone,ef1a:2xLynk:tdmSG(J) (F1 of allele 301),tdmScarlet3S2:H2B-tol2 (F1 of allele 324)
4,X:\abcabc\Aang_Foundation\20251106_mem-mito,/abcabc/Aang_Foundation/20251106_mem-mito,casper,ef1a:2xLynk:tdmChilada (F1 of allele 315)


In [88]:
# CELL L4 — merge ROI paths with imaging on mount_dir_norm
df_linked = df_roi_paths.merge(
    df_imaging,
    on="mount_dir_norm",
    how="left",
    suffixes=("", "_im"),
)

df_linked[[
    "roi_dir",
    "mount_dir_norm",
    "ZF female genotype",
    "ZF male genotype",
    "Data location"
]].head(20)

,roi_dir,mount_dir_norm,ZF female genotype,ZF male genotype,Data location
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250428_mem_histone/...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250428_mem_histon...
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250428_mem_histone/...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250428_mem_histon...
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,/abcabc/Korra_Foundation/20250429_mem_cytosol/...,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...


In [89]:
# CELL L5: Normalize parents

def norm_text(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    s = str(val).strip()
    return s or None

df_linked["parent_female"] = df_linked["ZF female genotype"].apply(norm_text)
df_linked["parent_male"]   = df_linked["ZF male genotype"].apply(norm_text)

df_linked[["roi_dir","parent_female","parent_male"]].head(20)

,roi_dir,parent_female,parent_male
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo


In [90]:
# CELL L7: Write structural ROI table (v7)

out_path = WORKING / "legacy_roi_structural_v2_mount_linked.csv"
df_linked.to_csv(out_path, index=False)
out_path, df_linked.shape

(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/legacy_roi_structural_v2_mount_linked.csv'),
 (1077, 30))

In [91]:
import pandas as pd
from pathlib import Path

BASE    = Path("/Users/davekokel/Projects/carp_v2")
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

struct_path = WORKING / "legacy_roi_structural_v7.csv"
sdf = pd.read_csv(struct_path)

sdf.loc[
    sdf["experiment_slug"] == "Korra_Foundation/20250428_mem_histone",
    ["roi_dir","experiment_slug","parent_female","parent_male"]
].head(10)

,roi_dir,experiment_slug,parent_female,parent_male
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Korra_Foundation/20250428_mem_histone,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Korra_Foundation/20250428_mem_histone,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
